### Dashboard summary tables - owners: Sweta, Omar Leopoldo, Shreyansh Pankaj

In [ ]:
monthly_revenue_gold = (
    order_gold.groupBy("YearMonth")
    .agg(
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.countDistinct("OrderID").alias("CompletedOrders"),
        F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
    ).orderBy("YearMonth")
)

category_performance_gold = (
    sales_line_gold.groupBy("Category")
    .agg(
        F.sum("Quantity").alias("UnitsSold"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.sum("GrossProfit"), 2).alias("GrossProfit"),
    ).orderBy(F.desc("NetRevenue"))
)

discount_impact_gold = (
    sales_line_gold
    .withColumn("DiscountBand", F.when(F.col("DiscountPct") == 0, "0%")
        .when(F.col("DiscountPct") <= 0.05, "5%")
        .when(F.col("DiscountPct") <= 0.10, "10%")
        .when(F.col("DiscountPct") <= 0.15, "15%")
        .otherwise("20%+"))
    .groupBy("DiscountBand")
    .agg(
        F.count("OrderItemID").alias("LineItems"),
        F.round(F.avg("Quantity"), 2).alias("AverageQuantity"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
    )
    .withColumn("SortOrder", F.when(F.col("DiscountBand") == "0%", 0)
        .when(F.col("DiscountBand") == "5%", 1)
        .when(F.col("DiscountBand") == "10%", 2)
        .when(F.col("DiscountBand") == "15%", 3).otherwise(4))
    .orderBy("SortOrder")
)

delivery_distance_gold = (
    order_gold.filter(F.col("ActualMinutes").isNotNull())
    .withColumn("DistanceBand", F.when(F.col("DistanceKm") <= 5, "0-5 km")
        .when(F.col("DistanceKm") <= 10, "5-10 km")
        .when(F.col("DistanceKm") <= 15, "10-15 km")
        .when(F.col("DistanceKm") <= 20, "15-20 km").otherwise("20+ km"))
    .groupBy("DistanceBand")
    .agg(
        F.countDistinct("OrderID").alias("Deliveries"),
        F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeRatePct"),
        F.round(F.avg("DelayMinutes"), 2).alias("AverageDelayMinutes"),
    )
    .withColumn("SortOrder", F.when(F.col("DistanceBand") == "0-5 km", 0)
        .when(F.col("DistanceBand") == "5-10 km", 1)
        .when(F.col("DistanceBand") == "10-15 km", 2)
        .when(F.col("DistanceBand") == "15-20 km", 3).otherwise(4))
    .orderBy("SortOrder")
)

loyalty_behavior_gold = (
    customer_behavior_gold.groupBy("LoyaltyStatus")
    .agg(
        F.countDistinct("CustomerID").alias("Customers"),
        F.round(F.avg("TotalOrders"), 2).alias("AverageOrders"),
        F.round(F.avg("AverageOrderValue"), 2).alias("AverageOrderValue"),
        F.round(F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2).alias("RepeatRatePct"),
    )
)

for name, df in [
    ("monthly_revenue", monthly_revenue_gold), ("category_performance", category_performance_gold),
    ("discount_impact", discount_impact_gold), ("delivery_distance", delivery_distance_gold),
    ("loyalty_behavior", loyalty_behavior_gold),
]:
    write_gold(df, name)